# Tutorial 2: Composition

This tutorial builds on [Tutorial 1 (counter)](counter.ipynb). We define an inverted pendulum environment and a controller neural network, then **compose** them into a closed-loop system using shared wires.

You'll learn:
1. How modules get their own fresh wires by default
2. How to create shared wire pairs and pass them to the `Env()` / `NN()` wrappers
3. How `Module.parallel` merges modules that share wires

## The inverted pendulum environment

An inverted pendulum balanced upright. The full nonlinear dynamics:

$$\ddot\theta = \frac{g}{\ell}\,\sin\theta + \frac{\tau}{m\ell^2}$$

The positive gravitational term makes the upright equilibrium ($\theta = 0$) **unstable** — without active control, any perturbation grows. The controller must apply torque $\tau$ to keep it balanced.

Write a standard `gym.Env`: `__init__` sets spaces and state, `reset` initializes, `step` updates. Then wrap with `Env()` to extract the symbolic module.

In [1]:
import math
import torch
import torch.nn as nn
import gymnasium as gym
from gymnasium import spaces
from zrth import Env, NN, Module


class InvertedPendulumEnv(gym.Env):
    """Inverted pendulum with full nonlinear dynamics."""

    def __init__(self, g=9.81, l=1.0, m=1.0, dt=0.05):
        super().__init__()

        self.action_space = spaces.Box(low=-2.0, high=2.0, shape=(1,))
        self.observation_space = spaces.Box(low=-10.0, high=10.0, shape=(1,))

        self.g = g
        self.l = l
        self.m = m
        self.dt = dt

        # Also set state variables here so the analyzer can infer their dtype
        self.theta = 0.02
        self.theta_dot = 0.0

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.theta = 0.02
        self.theta_dot = 0.0

        observation = self.theta
        return observation, {}

    def step(self, torque):
        accel = (self.g / self.l) * math.sin(self.theta) + torque / (self.m * self.l * self.l)
        self.theta_dot = self.theta_dot + accel * self.dt
        self.theta = self.theta + self.theta_dot * self.dt

        observation = self.theta
        reward = 0.0 - self.theta * self.theta - 0.1 * self.theta_dot * self.theta_dot
        terminated = self.theta > 3.14 or self.theta < -3.14
        truncated = False
        return observation, reward, terminated, truncated, {}

## Wrapping and composition

We create shared wire pairs up front so that the pendulum's observation feeds the controller's input, and the controller's output feeds the pendulum's action. Then we wrap both and compose them into a closed-loop system.

In [2]:
from zrth import Wire, Float

pendulum = InvertedPendulumEnv(g=9.81, l=1.0, m=1.0, dt=0.05)

# Shared wire pairs: pendulum observation = controller input
obs_wire = [Wire(Float(1)), Wire(Float(1))]
# Shared wire pairs: controller output = pendulum action
act_wire = [Wire(Float(1)), Wire(Float(1))]

wrapped_pendulum = Env(pendulum, observation=obs_wire, action=act_wire)
print(wrapped_pendulum)

module
  external
    w2, w3 : Float(1)
  interface
    w0, w1 : Float(1)
    w12, w13 : Float(1)
    w14, w15 : Bool(1)
    w16, w17 : Bool(1)
  private
    w4, w5 : Float(1)
    w6, w7 : Float(1)
  atom controls w1, w5, w7, w13, w15, w17 reads w2, w4, w6
  init
    Tensor([1]) w8; 
    Tensor([1]) w9; 
    Tensor([0.05000000074505806]) w10; 
    Tensor([9.8100004196167]) w11; 
    Tensor([0.019999999552965164]) w18; 
    Id w5; w18
    Tensor([0]) w19; 
    Id w7; w19
    Id w1; w18
    Tensor([0]) w13; 
    Const: false w15; 
    Const: false w17; 
  update
    Tensor([1]) w8; 
    Tensor([1]) w9; 
    Tensor([0.05000000074505806]) w10; 
    Tensor([9.8100004196167]) w11; 
    Div w20; w11, w9
    Sin w21; w4
    Mul w22; w20, w21
    Mul w23; w8, w9
    Mul w24; w23, w9
    Div w25; w2, w24
    Add w26; w22, w25
    Mul w27; w26, w10
    Add w28; w6, w27
    Id w7; w28
    Mul w29; w28, w10
    Add w30; w4, w29
    Id w5; w30
    Tensor([0]) w31; 
    Mul w32; w30, w30
    Sub w33;

A small NN that observes $\theta$ and outputs a torque $\tau$. Architecture: $[1] \to 4 \to [1]$. We wrap it with the same shared wires.

In [3]:
class ControllerNN(nn.Module):
    """NN controller: theta -> torque. Uses tanh for symmetric output."""

    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(1, 16)
        self.fc2 = nn.Linear(16, 1)

    def forward(self, state):
        x = torch.tanh(self.fc1(state))
        return 2.0 * torch.tanh(self.fc2(x))  # scale to [-2, 2]

In [4]:
controller = ControllerNN()
wrapped_controller = NN(controller, extl=obs_wire, intf=act_wire)
print(wrapped_controller)

module
  external
    w0, w1 : Float(1)
  interface
    w2, w3 : Float(1)
  atom controls w3 awaits w1
  init
    Tensor([-0.001135110855102539 -0.5820412635803223 0.5753635168075562 -0.16961669921875 -0.29929375648498535 ...]) w48; 
    Tensor([0.03514707088470459 0.9030269384384155 0.25593292713165283 0.5230144262313843 -0.7023398876190186 ...]) w49; 
    Linear w50; w1, w48, w49
    Tanh w51; w50
    Tensor([2]) w52; 
    Tensor([-0.13166528940200806 0.21339759230613708 0.08049100637435913 -0.18090087175369263 0.10891222953796387 ...]) w53; 
    Tensor([0.04606097936630249]) w54; 
    Linear w55; w51, w53, w54
    Tanh w56; w55
    Mul w57; w52, w56
    Id w3; w57
  update
    Tensor([-0.001135110855102539 -0.5820412635803223 0.5753635168075562 -0.16961669921875 -0.29929375648498535 ...]) w48; 
    Tensor([0.03514707088470459 0.9030269384384155 0.25593292713165283 0.5230144262313843 -0.7023398876190186 ...]) w49; 
    Linear w50; w1, w48, w49
    Tanh w51; w50
    Tensor([2]) w52; 


In [5]:
closed_loop = Env(wrapped_pendulum, wrapped_controller)
print(closed_loop)

module
  interface
    w0, w1 : Float(1)
    w12, w13 : Float(1)
    w14, w15 : Bool(1)
    w16, w17 : Bool(1)
    w2, w3 : Float(1)
  private
    w4, w5 : Float(1)
    w6, w7 : Float(1)
  atom controls w1, w5, w7, w13, w15, w17 reads w2, w4, w6
  init
    Tensor([1]) w8; 
    Tensor([1]) w9; 
    Tensor([0.05000000074505806]) w10; 
    Tensor([9.8100004196167]) w11; 
    Tensor([0.019999999552965164]) w18; 
    Id w5; w18
    Tensor([0]) w19; 
    Id w7; w19
    Id w1; w18
    Tensor([0]) w13; 
    Const: false w15; 
    Const: false w17; 
  update
    Tensor([1]) w8; 
    Tensor([1]) w9; 
    Tensor([0.05000000074505806]) w10; 
    Tensor([9.8100004196167]) w11; 
    Div w20; w11, w9
    Sin w21; w4
    Mul w22; w20, w21
    Mul w23; w8, w9
    Mul w24; w23, w9
    Div w25; w2, w24
    Add w26; w22, w25
    Mul w27; w26, w10
    Add w28; w6, w27
    Id w7; w28
    Mul w29; w28, w10
    Add w30; w4, w29
    Id w5; w30
    Tensor([0]) w31; 
    Mul w32; w30, w30
    Sub w33; w31, w32
 

## Training the controller

We train the controller using **REINFORCE** (policy gradient). The wrapped objects are fully functional — `wrapped_pendulum.reset()` / `wrapped_pendulum.step()` work as gym methods, and `wrapped_controller(obs)` runs a PyTorch forward pass with autograd.

The controller outputs a mean torque $\mu$; we sample from $\mathcal{N}(\mu, \sigma^2)$ to explore. The loss is the standard REINFORCE estimator:

$$\nabla J = \mathbb{E}\left[\sum_t \nabla \log \pi(a_t | s_t) \cdot G_t\right]$$

where $G_t$ is the discounted return from step $t$.

Because `NN` uses live tensor references, the symbolic module and the composed `closed_loop` automatically reflect the trained weights — no re-wrapping needed.

In [6]:
def compute_returns(rewards, gamma=0.99):
    """Compute discounted returns for each timestep."""
    returns = []
    G = 0.0
    for r in reversed(rewards):
        G = r + gamma * G
        returns.insert(0, G)
    return returns

optimizer = torch.optim.Adam(wrapped_controller.parameters(), lr=0.005)
n_episodes = 1000
max_steps = 200
gamma = 0.99

for episode in range(n_episodes):
    wrapped_pendulum.reset()
    log_probs = []
    rewards = []

    sigma = max(0.3, 1.0 - episode / 500)  # decay exploration noise

    for step in range(max_steps):
        state = torch.tensor([wrapped_pendulum.theta], dtype=torch.float32)
        mu = wrapped_controller(state.unsqueeze(0)).squeeze()
        dist = torch.distributions.Normal(mu, sigma)
        action = dist.sample().clamp(-2.0, 2.0)
        log_probs.append(dist.log_prob(action))

        _, reward, terminated, _, _ = wrapped_pendulum.step(action.item())
        rewards.append(reward + 0.1)  # survival bonus
        if terminated:
            break

    # Policy gradient update
    returns = compute_returns(rewards, gamma)
    returns = torch.tensor(returns, dtype=torch.float32)
    returns = (returns - returns.mean()) / (returns.std() + 1e-8)  # baseline

    loss = torch.tensor(0.0)
    for lp, G in zip(log_probs, returns):
        loss = loss - lp * G

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if episode % 100 == 0:
        total_reward = sum(rewards)
        print(f"episode {episode:3d}  steps = {len(rewards):3d}  reward = {total_reward:.2f}  sigma = {sigma:.2f}")

episode   0  steps =  35  reward = -79.10  sigma = 1.00
episode 100  steps =  35  reward = -74.59  sigma = 0.80
episode 200  steps =  35  reward = -67.98  sigma = 0.60
episode 300  steps =  37  reward = -73.36  sigma = 0.40
episode 400  steps =  42  reward = -73.09  sigma = 0.30
episode 500  steps =  33  reward = -81.31  sigma = 0.30
episode 600  steps =  31  reward = -65.77  sigma = 0.30
episode 700  steps =  39  reward = -78.81  sigma = 0.30
episode 800  steps =  39  reward = -76.74  sigma = 0.30
episode 900  steps =  42  reward = -78.95  sigma = 0.30


## Evaluating the trained controller

Run the trained controller deterministically (no exploration noise) and print $\theta$ at each step.

In [7]:
wrapped_pendulum.reset()

print(f"{'step':>4}  {'theta':>8}  {'theta_dot':>10}  {'torque':>8}  {'reward':>8}")
print("-" * 50)

with torch.no_grad():
    for step in range(20):
        state = torch.tensor([wrapped_pendulum.theta], dtype=torch.float32)
        torque = wrapped_controller(state.unsqueeze(0)).squeeze().item()
        torque = max(-2.0, min(2.0, torque))

        _, reward, terminated, _, _ = wrapped_pendulum.step(torque)
        print(f"{step:4d}  {wrapped_pendulum.theta:8.4f}  {wrapped_pendulum.theta_dot:10.4f}  {torque:8.4f}  {reward:.4f}")
        if terminated:
            print("terminated!")
            break

step     theta   theta_dot    torque    reward
--------------------------------------------------
   0    0.0196     -0.0084   -0.3637  -0.0004
   1    0.0187     -0.0169   -0.3633  -0.0004
   2    0.0174     -0.0259   -0.3624  -0.0004
   3    0.0157     -0.0354   -0.3611  -0.0004
   4    0.0134     -0.0456   -0.3594  -0.0004
   5    0.0105     -0.0569   -0.3571  -0.0004
   6    0.0071     -0.0695   -0.3542  -0.0005
   7    0.0029     -0.0835   -0.3507  -0.0007
   8   -0.0021     -0.0994   -0.3465  -0.0010
   9   -0.0080     -0.1175   -0.3415  -0.0014
  10   -0.0149     -0.1382   -0.3356  -0.0021
  11   -0.0230     -0.1619   -0.3286  -0.0032
  12   -0.0324     -0.1892   -0.3203  -0.0046
  13   -0.0435     -0.2207   -0.3105  -0.0068
  14   -0.0563     -0.2569   -0.2991  -0.0098
  15   -0.0712     -0.2988   -0.2857  -0.0140
  16   -0.0886     -0.3472   -0.2700  -0.0199
  17   -0.1088     -0.4032   -0.2515  -0.0281
  18   -0.1322     -0.4679   -0.2298  -0.0394
  19   -0.1593     -0.5428  

## Running the composed closed-loop system

The `closed_loop` is an `Env` — it composes the pendulum and controller into a single Module, while keeping the real pendulum env for delegation. When we call `step()`, the pendulum atom runs the real env (for rendering, real physics) and the controller atom runs symbolically. Outputs flow between them via shared wires.

Because the symbolic module holds live tensor references to the controller's weights, the composed module already reflects the trained values — no re-wrapping needed.

In [8]:
closed_loop.reset()

print(f"{'step':>4}  {'theta':>8}  {'theta_dot':>10}")
print("-" * 30)

for step in range(20):
    obs, reward, terminated, truncated, info = closed_loop.step(0)
    print(f"{step:4d}  {closed_loop.theta:8.4f}  {closed_loop.theta_dot:10.4f}")
    if terminated:
        print("terminated!")
        break

step     theta   theta_dot
------------------------------
   0    0.0205      0.0098
   1    0.0215      0.0199
   2    0.0230      0.0304
   3    0.0251      0.0417
   4    0.0278      0.0540
   5    0.0312      0.0676
   6    0.0353      0.0829
   7    0.0403      0.1002
   8    0.0463      0.1200
   9    0.0535      0.1427
  10    0.0619      0.1689
  11    0.0719      0.1992
  12    0.0836      0.2345
  13    0.0974      0.2754
  14    0.1135      0.3231
  15    0.1324      0.3787
  16    0.1546      0.4434
  17    0.1806      0.5190
  18    0.2109      0.6071
  19    0.2464      0.7097


## Verifying execution order and correctness

The composed module has two atoms. The topological sort orders them by **await dependencies**: the controller `awaits` the pendulum's observation wire, so the pendulum atom executes first. We can inspect this directly.

We also validate that `Env` (which runs the real pendulum env for the env atom and the symbolic interpreter for the controller atom) produces identical results to `Simulator` (which runs both atoms symbolically).

In [ ]:
# Execution order: inspect the atoms and their await dependencies
for idx in range(len(closed_loop.atoms)):
    atom = closed_loop.atoms[idx]
    n_ctrl = len(atom.ctrl)
    n_wait = len(atom.wait)
    ctrl_ids = [atom.ctrl[j].id for j in range(n_ctrl)]
    wait_ids = [atom.wait[j].id for j in range(n_wait)]

    is_env = idx == closed_loop._env_atom_idx
    label = "pendulum (real env)" if is_env else "controller (symbolic)"

    print(f"Atom {idx}: {label}")
    print(f"  controls: {ctrl_ids}")
    print(f"  awaits:   {wait_ids}")
    print()

In [ ]:
import numpy as np
from zrth import Simulator

# Compare Env (real env + symbolic controller) vs Simulator (pure symbolic)
sim_closed = Simulator(wrapped_pendulum, wrapped_controller)

closed_loop.reset()
sim_closed.reset()

print(f"{'step':>4}  {'Env theta':>12}  {'Sim theta':>12}  {'match':>8}")
print("-" * 45)

all_match = True
for step in range(20):
    closed_loop.step(0)
    sim_closed.step(np.zeros(1))

    env_theta = closed_loop.theta
    sim_theta = float(sim_closed._state[sim_closed._observation_pair[0].id].item())
    match = abs(env_theta - sim_theta) < 1e-6
    all_match = all_match and match
    print(f"{step:4d}  {env_theta:12.6f}  {sim_theta:12.6f}  {'OK' if match else 'MISMATCH':>8}")

print()
print(f"All steps match: {all_match}")